# 01. EDA（探索的データ分析）
2022年中央競馬データの基本統計・分布・相関を確認し、回収率に寄与しうる特徴を探索する。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/processed/features.csv', encoding='utf-8-sig')
print(f'shape: {df.shape}')
print(f'レース数: {df["race_id"].nunique()}')
df.head()

## 1. 着順・人気の分布

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df['確定着順'].value_counts().sort_index().plot(kind='bar', ax=axes[0], title='Finish Position Distribution')
axes[0].set_xlabel('Finish Position')

df['単勝人気'].value_counts().sort_index().plot(kind='bar', ax=axes[1], title='Popularity Distribution', color='orange')
axes[1].set_xlabel('Popularity (Betting Rank)')

df['単勝オッズ'].clip(upper=50).hist(bins=50, ax=axes[2], edgecolor='black')
axes[2].set_title('Win Odds Distribution (clipped at 50)')
axes[2].set_xlabel('Win Odds')

plt.tight_layout()
plt.show()

## 2. 人気別 勝率・回収率

In [ ]:
import sys; sys.path.append('../src')
from evaluate_roi import roi_by_bet_condition

rows = []
for pop in sorted(df['単勝人気'].unique()):
    grp = df[df['単勝人気'] == pop]
    mask = pd.Series(True, index=grp.index)
    r = roi_by_bet_condition(grp, mask, bet_col='payout_win')
    r['popularity'] = pop
    rows.append(r)
pop_df = pd.DataFrame(rows).set_index('popularity')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pop_df['hit_rate'][:16].plot(kind='bar', ax=axes[0], title='Win Rate by Popularity (%)')
pop_df['roi'][:16].plot(kind='bar', ax=axes[1], title='Win Return Rate by Popularity (%)', color='tomato')
for ax in axes:
    ax.set_xlabel('Popularity')
axes[1].axhline(100, color='black', linestyle='--', linewidth=1, label='break-even')
axes[1].legend()
plt.tight_layout()
plt.show()

print(pop_df[['n_bets','hit_rate','roi']].head(16).to_string())

## 3. 得点・予想タイム指数と着順の関係

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, label in zip(axes,
    ['得点', '予想タイム指数', '騎手評価'],
    ['Score', 'Estimated Time Index', 'Jockey Rating']):
    sub = df[df['確定着順'] <= 10]
    sub.boxplot(column=col, by='確定着順', ax=ax)
    ax.set_title(f'{label} vs Finish Pos')
    ax.set_xlabel('Finish Position')
    plt.sca(ax); plt.title(f'{label} vs Finish Pos')

plt.suptitle('')
plt.tight_layout()
plt.show()

## 4. 馬体重変化 vs 勝敗

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df.boxplot(column='馬体重増減', by='label_win', ax=axes[0])
axes[0].set_title('Weight Change vs Win (0=lose/1=win)')
axes[0].set_xlabel('label_win')
plt.sca(axes[0]); plt.title('Weight Change vs Win')

bins = [-50, -10, -4, 0, 4, 10, 50]
labels = ['<-10','-10~-4','-4~0','0~4','4~10','>10']
df['weight_bin'] = pd.cut(df['馬体重増減'], bins=bins, labels=labels)
rows = []
for b, grp in df.groupby('weight_bin', observed=True):
    r = roi_by_bet_condition(grp, pd.Series(True, index=grp.index), bet_col='payout_win')
    r['bin'] = b
    rows.append(r)
wdf = pd.DataFrame(rows).set_index('bin')
wdf['roi'].plot(kind='bar', ax=axes[1], title='ROI by Weight Change', color='steelblue')
axes[1].axhline(100, color='red', linestyle='--')
axes[1].set_xlabel('Weight Change (kg)')

plt.suptitle('')
plt.tight_layout()
plt.show()
print(wdf[['n_bets','hit_rate','roi']])

## 5. 距離・馬場別 回収率

In [ ]:
rows = []
for (track, dist_cat), grp in df.groupby(
    ['track_label', pd.cut(df['距離'], bins=[0,1400,1800,2200,9999], labels=['Sprint','Mile','Middle','Long'])],
    observed=True):
    r = roi_by_bet_condition(grp, pd.Series(True, index=grp.index), bet_col='payout_win')
    r['track'] = track
    r['dist_cat'] = dist_cat
    rows.append(r)
dist_df = pd.DataFrame(rows)
pivot = dist_df.pivot(index='dist_cat', columns='track', values='roi')
pivot.plot(kind='bar', title='ROI by Track x Distance (%)')
plt.axhline(100, color='black', linestyle='--')
plt.xlabel('Distance Category')
plt.tight_layout()
plt.show()
print(pivot)

## 6. 馬場状態別 回収率

In [ ]:
rows = []
for cond, grp in df.groupby('cond_label'):
    r = roi_by_bet_condition(grp, pd.Series(True, index=grp.index), bet_col='payout_win')
    r['condition'] = cond
    rows.append(r)
cond_df = pd.DataFrame(rows).set_index('condition')
cond_df[['hit_rate','roi']].plot(kind='bar', title='Win Rate and ROI by Track Condition')
plt.axhline(100, color='gray', linestyle='--')
plt.tight_layout()
plt.show()
print(cond_df[['n_bets','hit_rate','roi']])

## 7. 予想オッズ vs 実際のオッズ（割安馬の存在確認）

In [ ]:
sample = df.sample(3000, random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(sample['予想オッズ'], sample['単勝オッズ'],
                c=sample['label_win'], cmap='RdYlGn', alpha=0.3, s=10)
lim = min(sample['予想オッズ'].max(), sample['単勝オッズ'].max(), 100)
axes[0].plot([0, lim], [0, lim], 'k--', linewidth=0.8)
axes[0].set_xlim(0, lim); axes[0].set_ylim(0, lim)
axes[0].set_xlabel('Predicted Odds'); axes[0].set_ylabel('Actual Odds')
axes[0].set_title('Predicted vs Actual Odds (green=winner)')

df['log_odds_gap'].clip(-3, 3).hist(bins=60, ax=axes[1], edgecolor='black')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('Log Odds Gap Distribution (>0 = undervalued)')
axes[1].set_xlabel('log(actual_odds) - log(predicted_odds)')

plt.tight_layout()
plt.show()

## 8. 主要数値特徴量の相関ヒートマップ

In [ ]:
num_cols = ['単勝オッズ','単勝人気','log_odds','odds_gap','prob_norm',
            '得点','予想タイム指数','騎手評価','調教師評価','枠順評価','脚質評価',
            '馬体重','馬体重増減','馬齢','距離','cond_code',
            '先行指数','血統総合評価','波乱度','レースレベル',
            '確定着順','label_win','label_place']
corr = df[num_cols].corr()

plt.figure(figsize=(15, 12))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.3, annot_kws={'size': 7})
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 9. 着順別 主要スコアの平均値比較

In [ ]:
score_cols = ['得点','予想タイム指数','得点V1','得点V2','得点V3',
              '騎手評価','調教師評価','先行指数','血統総合評価']
mean_by_pos = df[df['確定着順'] <= 6].groupby('確定着順')[score_cols].mean()

# 正規化して比較
mean_norm = (mean_by_pos - mean_by_pos.min()) / (mean_by_pos.max() - mean_by_pos.min())
mean_norm.T.plot(kind='bar', figsize=(14, 5),
                 title='Normalized Score by Finish Position (1st to 6th)')
plt.xlabel('Feature')
plt.ylabel('Normalized Score (0=min, 1=max)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()